# Exercise 5 - Transfer learning

> **GPU: Runtime -> Change runtime type -> T4 GPU.** About 5 minutes.

Same ants/bees data, but a **different backbone**: `mobilenet_v3_small`. Its head is not called
`fc`, its normalization comes from the weights object, and it is small enough that the whole
exercise runs fast. Finding the head of an unfamiliar model is half the skill.

Seven tasks.

In [ ]:
import time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device', device)
if device.type != 'cuda':
    print('*** no GPU: switch the runtime, or expect this to be slow ***')

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
ROOT = Path('/content' if IN_COLAB else '.')
NUM_WORKERS = 2 if IN_COLAB else 0

def set_seed(seed=0):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(0)
torch.backends.cudnn.benchmark = True
plt.rcParams['figure.dpi'] = 110

data_root = ROOT / 'hymenoptera_data'
if not data_root.exists():
    urllib.request.urlretrieve('https://download.pytorch.org/tutorial/hymenoptera_data.zip',
                               ROOT / 'hymenoptera_data.zip')
    with zipfile.ZipFile(ROOT / 'hymenoptera_data.zip') as z:
        z.extractall(ROOT)
print('data at', data_root)

WEIGHTS = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
print('backbone: mobilenet_v3_small', WEIGHTS)

---
## Task 1 - Get the preprocessing from the weights

Don't hand-copy normalization constants. Every torchvision weights enum carries the exact
transform it was trained with.

1. `eval_tf` - obtained from `WEIGHTS` itself (one call).
2. `train_tf` - your own augmentation pipeline, ending in the **same** normalization as `eval_tf`.
   Read the mean/std off the weights metadata rather than typing them in.

In [ ]:
# TODO: eval_tf = ...    (hint: WEIGHTS has a .transforms() method)
# TODO: read the mean and std that eval_tf uses, into MEAN and STD
# TODO: train_tf = transforms.Compose([...]) with RandomResizedCrop(224), a flip,
#       ToTensor, and Normalize(MEAN, STD)

print('eval_tf:', eval_tf)
assert callable(eval_tf)
assert list(MEAN) == [0.485, 0.456, 0.406], f'MEAN should be the ImageNet stats, got {MEAN}'
assert list(STD) == [0.229, 0.224, 0.225], f'STD should be the ImageNet stats, got {STD}'

train_ds = datasets.ImageFolder(data_root / 'train', train_tf)
val_ds = datasets.ImageFolder(data_root / 'val', eval_tf)
CLASSES = train_ds.classes

x_tr, _ = train_ds[0]
x_va, _ = val_ds[0]
assert x_tr.shape == (3, 224, 224), f'train sample {tuple(x_tr.shape)}'
assert x_va.shape == (3, 224, 224), f'val sample {tuple(x_va.shape)}'
assert any('Random' in type(t).__name__ for t in train_tf.transforms), 'train_tf needs augmentation'
print(f'PASS  classes {CLASSES} | train {len(train_ds)} | val {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=device.type == 'cuda')
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=device.type == 'cuda')

def denormalize(t):
    m = torch.tensor(MEAN).view(-1, 1, 1); s = torch.tensor(STD).view(-1, 1, 1)
    return (t.detach().cpu() * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(denormalize(xb[i])); ax.set_title(CLASSES[yb[i]], fontsize=8); ax.axis('off')
plt.tight_layout()

---
## Task 2 - Find and replace the head

`mobilenet_v3_small` does **not** have `.fc`. Print the model, find the final classifier layer,
and write `replace_head(model, n_classes)` that swaps it for a fresh `nn.Linear` with the right
`in_features` - read from the existing layer, never hard-coded.

The assertions check the rest of the network is untouched.

In [ ]:
probe = models.mobilenet_v3_small(weights=WEIGHTS)
print('classifier:', probe.classifier)
print('\ntop-level children:', [n for n, _ in probe.named_children()])

def replace_head(model, n_classes):
    """Swap the final classification layer for a fresh Linear(in_features, n_classes). Returns the model."""
    # TODO
    raise NotImplementedError


m = replace_head(models.mobilenet_v3_small(weights=WEIGHTS), 2)
ref = models.mobilenet_v3_small(weights=WEIGHTS)

out = m(torch.randn(2, 3, 224, 224))
assert out.shape == (2, 2), f'output {tuple(out.shape)} should be (2, 2)'
last = [mod for mod in m.classifier if isinstance(mod, nn.Linear)][-1]
assert last.out_features == 2, 'the final Linear must have 2 outputs'
assert last.in_features == [mod for mod in ref.classifier if isinstance(mod, nn.Linear)][-1].in_features, \
    'in_features must match the original layer - read it, do not hard-code'
assert torch.equal(m.features[0][0].weight, ref.features[0][0].weight), 'the backbone must be unchanged'
print(f'PASS  new head: Linear({last.in_features}, {last.out_features})')

---
## Task 3 - Freeze the backbone

Write `make_frozen(n_classes)` returning a model where:

- every backbone parameter has `requires_grad=False`
- the new head is trainable
- fewer than 5,000 trainable parameters (the head is `Linear(1024, 2)`, so ~2k)

Order matters: freeze first, *then* replace the head (new layers default to trainable). Do it the
other way round and you freeze your own head too.

In [ ]:
def make_frozen(n_classes=2):
    """-> model with a frozen backbone and a fresh trainable head."""
    # TODO
    raise NotImplementedError


fm = make_frozen(2)
trainable = [n for n, p in fm.named_parameters() if p.requires_grad]
n_train = sum(p.numel() for p in fm.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in fm.parameters())

assert n_train < 5000, f'{n_train:,} trainable parameters - is the backbone really frozen?'
assert n_train > 0, 'nothing is trainable - you froze the head too'
assert all('classifier' in n for n in trainable), f'only the head should train, got {trainable}'
assert fm(torch.randn(1, 3, 224, 224)).shape == (1, 2)
print(f'PASS  trainable {n_train:,} of {n_all:,} ({100 * n_train / n_all:.3f}%)')
print('      trainable tensors:', trainable)

---
## Task 4 - Train the frozen model

Reuse the loop from chapter 4. Pass **only the trainable parameters** to the optimizer.

Target: **val accuracy > 0.90** in 5 epochs. (This is a 2-class problem, so chance is 0.50.)

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, device, scheduler=None):
    # TODO
    raise NotImplementedError

@torch.no_grad()
def evaluate(model, loader, device, return_preds=False):
    # TODO: -> (loss, acc) or (loss, acc, y_true, y_pred, y_prob)
    raise NotImplementedError

def run(model, optimizer, epochs, scheduler=None, label=''):
    """Train for `epochs`, return (history, best_val_acc, best_state_dict)."""
    # TODO
    raise NotImplementedError


EPOCHS = 5
set_seed(0)
model_frozen = make_frozen(2).to(device)
opt = torch.optim.SGD([p for p in model_frozen.parameters() if p.requires_grad],
                      lr=0.02, momentum=0.9, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

hist_frozen, best_frozen, _ = run(model_frozen, opt, EPOCHS, sch, label='frozen')
assert best_frozen > 0.90, f'best val accuracy {best_frozen:.4f} below 0.90'
assert len(hist_frozen['val_acc']) == EPOCHS
print(f'PASS  frozen backbone: best val accuracy {best_frozen:.4f}')

---
## Task 5 - Fine-tune with discriminative learning rates

Build `make_finetune_optimizer(model, backbone_lr, head_lr)` returning an SGD optimizer with
**two parameter groups**: the backbone at `backbone_lr` and the head at `head_lr`.

Then train a fully unfrozen model with `backbone_lr=1e-3`, `head_lr=1e-2` and beat the frozen
result (or match it - on this tiny dataset the frozen model is already strong).

In [ ]:
def make_finetune_optimizer(model, backbone_lr, head_lr, momentum=0.9, weight_decay=1e-4):
    """SGD with two param groups. Group 0 = backbone at backbone_lr, group 1 = head at head_lr."""
    # TODO
    raise NotImplementedError


set_seed(0)
model_ft = replace_head(models.mobilenet_v3_small(weights=WEIGHTS), 2).to(device)
opt_ft = make_finetune_optimizer(model_ft, 1e-3, 1e-2)

assert len(opt_ft.param_groups) == 2, 'need exactly two parameter groups'
assert opt_ft.param_groups[0]['lr'] == 1e-3 and opt_ft.param_groups[1]['lr'] == 1e-2
n_g0 = sum(p.numel() for p in opt_ft.param_groups[0]['params'])
n_g1 = sum(p.numel() for p in opt_ft.param_groups[1]['params'])
assert n_g0 > n_g1 * 100, f'group 0 should be the big backbone ({n_g0:,} vs {n_g1:,})'
assert n_g0 + n_g1 == sum(p.numel() for p in model_ft.parameters()), 'every parameter must be in some group'

sch_ft = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=EPOCHS)
hist_ft, best_ft, best_state_ft = run(model_ft, opt_ft, EPOCHS, sch_ft, label='fine-tune')
assert best_ft > 0.90, f'best {best_ft:.4f}'
print(f'PASS  fine-tuned: best val accuracy {best_ft:.4f} (frozen was {best_frozen:.4f})')

plt.figure(figsize=(6, 3.6))
plt.plot(hist_frozen['val_acc'], marker='o', label=f'frozen ({best_frozen:.3f})')
plt.plot(hist_ft['val_acc'], marker='s', label=f'fine-tuned ({best_ft:.3f})')
plt.axhline(0.5, ls=':', c='k', label='chance')
plt.xlabel('epoch'); plt.ylabel('val accuracy'); plt.legend(fontsize=8); plt.grid(alpha=0.3)

---
## Task 6 - Prove that pretraining is what's doing the work

Train the *same* architecture with `weights=None` (random init), same optimizer settings, same
epochs. It should be dramatically worse.

Then answer: why can't more epochs fix this?

In [ ]:
# TODO: build model_scratch with weights=None and a 2-class head, train it for EPOCHS,
#       store best accuracy in best_scratch

set_seed(0)
raise NotImplementedError

print(f'{"strategy":18} {"best val acc":>13}')
print(f'{"from scratch":18} {best_scratch:13.4f}')
print(f'{"frozen backbone":18} {best_frozen:13.4f}')
print(f'{"fine-tuned":18} {best_ft:13.4f}')
assert best_frozen > best_scratch + 0.05, 'pretraining should win clearly on 244 images'
print(f'\nPASS  pretraining is worth {best_frozen - best_scratch:+.3f} accuracy here')

**Why can't more epochs fix the from-scratch model?** ...

---
## Task 7 - Grad-CAM

Implement Grad-CAM for the fine-tuned model and plot 6 validation images with their heatmaps.

Steps: hook the last conv feature maps and their gradients, backprop from one class score,
weight each channel by its mean gradient, sum, ReLU, upsample, normalize to 0-1.

For `mobilenet_v3_small` the target layer is `model.features[-1]`.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        # TODO: register a forward hook to save activations and a full backward hook for gradients
        raise NotImplementedError

    def __call__(self, x, class_idx=None):
        """-> (cam as (H, W) numpy in 0..1, predicted class index, confidence)"""
        # TODO
        raise NotImplementedError

    def close(self):
        # TODO: remove the hooks
        raise NotImplementedError


model_ft.load_state_dict(best_state_ft)
cam_engine = GradCAM(model_ft.eval(), model_ft.features[-1])
val_plain = datasets.ImageFolder(data_root / 'val', None)

test_x = eval_tf(val_plain[0][0])[None].to(device)
cam, pred, conf = cam_engine(test_x)
assert np.asarray(cam).shape == (224, 224), f'cam shape {np.asarray(cam).shape} should be (224, 224)'
assert 0.0 <= cam.min() and cam.max() <= 1.0 + 1e-6, 'normalize the cam to 0..1'
assert abs(cam.max() - 1.0) < 1e-5, 'the max of a normalized cam should be 1.0'
assert pred in (0, 1) and 0.0 <= conf <= 1.0
print(f'PASS  cam {cam.shape} | pred {CLASSES[pred]} conf {conf:.3f}')

# TODO: plot 6 images with their Grad-CAM overlays, titled with true/pred labels
cam_engine.close()

**Looking at your heatmaps: is the model using the insect, or the background?** ...

---
## Done

- [ ] I can find and replace the classifier head of a model I've never seen.
- [ ] I know why the backbone gets a lower learning rate.
- [ ] I know that `requires_grad=False` does not freeze BatchNorm statistics.
- [ ] I can write Grad-CAM from the four-step description.

Solutions: [`solutions/sol05_transfer.ipynb`](solutions/sol05_transfer.ipynb)